# Exploratory Data Analysis: Exploring relationships between air quality, mobility outcomes, and neighborhood characteristics.
First let's start with imports and loading the data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
characteristics = pd.read_csv('characteristics_clean.csv')
mobility = pd.read_csv('mobility.csv')
pm_concentrations = pd.read_csv('air_quality_short.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'characteristics_clean.csv'

Let's compare the info about our dataframes for characteristics and mobility to see if there are any discrepencies before we merge them.


In [ ]:
characteristics.info()

In [ ]:
mobility.info()

There appears to be two counties missing from our mobility dataset represented by a difference in 2 rows. These 2 rows were dropped earlier during pre-processing of mobility outcomes due to containing NA values. Let's find out what counties those were:

In [ ]:
missing = set(characteristics['county']) - set(mobility['county'])
print(f"The two counties that are missing from mobility are: {missing}")

Now let's go ahead and merge our datasets together and see if there are relationships between neighborhood characteristics and mobility outcomes that could act as confounders when analyzing the relationship between air quality and outcomes.

In [ ]:
county_data = pd.merge(characteristics, mobility, on='county')
county_data.head(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Plot 1: Individual Income vs. Median Household Income
sns.scatterplot(data=county_data, x='Median_Household_Income_1990',
                y='Individual_Income_Excluding_Spouse_at_Age_35', ax=axes[0, 0])
axes[0, 0].set_title('Individual Income vs. Median Household Income')
axes[0, 0].set_xlabel('Median Household Income (1990)')
axes[0, 0].set_ylabel('Individual Income at Age 35')

# Plot 2: College Grad Rates vs. Median Household Income
sns.scatterplot(data=county_data, x='Median_Household_Income_1990',
                y='College_Grad_Rate', ax=axes[0, 1])
axes[0, 1].set_title('College Grad Rate vs. Median Household Income')
axes[0, 1].set_xlabel('Median Household Income (1990)')
axes[0, 1].set_ylabel('College Graduation Rate')

# Plot 3: Individual Income vs. Fraction Nonwhite
sns.scatterplot(data=county_data, x='Fraction_Nonwhite_2010',
                y='Individual_Income_Excluding_Spouse_at_Age_35', ax=axes[0, 2])
axes[0, 2].set_title('Individual Income vs. Fraction Nonwhite')
axes[0, 2].set_xlabel('Fraction Nonwhite (2010)')
axes[0, 2].set_ylabel('Individual Income at Age 35')

# Plot 4: College Grad Rates vs. Fraction Nonwhite
sns.scatterplot(data=county_data, x='Fraction_Nonwhite_2010',
                y='College_Grad_Rate', ax=axes[1, 0])
axes[1, 0].set_title('College Grad Rate vs. Fraction Nonwhite')
axes[1, 0].set_xlabel('Fraction Nonwhite (2010)')
axes[1, 0].set_ylabel('College Graduation Rate')

# Plot 5: Individual Income vs. Avg Job Growth
sns.scatterplot(data=county_data, x='Avg_Job_Growth_2004_2013',
                y='Individual_Income_Excluding_Spouse_at_Age_35', ax=axes[1, 1])
axes[1, 1].set_title('Individual Income vs. Avg Job Growth')
axes[1, 1].set_xlabel('Average Job Growth (2004-2013)')
axes[1, 1].set_ylabel('Individual Income at Age 35')

# Plot 6: College Grad Rates vs. Avg Job Growth
sns.scatterplot(data=county_data, x='Avg_Job_Growth_2004_2013',
                y='College_Grad_Rate', ax=axes[1, 2])
axes[1, 2].set_title('College Grad Rate vs. Avg Job Growth')
axes[1, 2].set_xlabel('Average Job Growth (2004-2013)')
axes[1, 2].set_ylabel('College Graduation Rate')

plt.tight_layout()
plt.show()

## Interpretting the neighborhood characteristics vs outcomes plots
Based on the graphs, there is a slight notion of a positive relationship between fraction nonwhite and outcomes, but it appears noisy or neglible. This could be due to fraction nonwhite being collected in counties in 2010, 30 years after the birth of our surveyed participants and/or a lack of causality and influence between the two variables in general. The graph of average job growth in counties and their primary outcomes appears to indicate an even stronger notion, although still noisy and possibly negligible. A strong positive correlation appears to be present between median household income and opportunity outcomes.

Based on these observations, median household income is a neighborhood characteristic we may wish to consider when exploring the relationship between air quality and outcomes.

Now let's take a look at air quality and compare air quality to neighborhood characteristics. Is there a relationship?

In [ ]:
pm_concentrations.head(5)

In [ ]:
year_data = pm_concentrations[pm_concentrations['year'] == 2001]
data = pd.merge(county_data, year_data, on='county')
sns.scatterplot(data=data, x='Median_Household_Income_1990',
                y='pm25_mean_median')
plt.title('Median Household Income vs Air Quality')
plt.xlabel('Median Household Income')
plt.ylabel('Air Quality (PM25 Mean Median)')
plt.show()

In [ ]:
# Create income bins to categorize median household income
income_bins = [0, 20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000]  # Adjust bins based on your data distribution
income_labels = ['0-20k', '20k-40k', '40k-60k', '60k-80k', '80k-100k', '100k-120k', '120k-140k', '140k+']
data['Income_Bin'] = pd.cut(data['Median_Household_Income_1990'], bins=income_bins, labels=income_labels)

# Create a box plot showing the distribution of air quality (PM25) across income bins
plt.figure(figsize=(10, 6))
sns.boxplot(data=data, x='Income_Bin', y='pm25_mean_median')
plt.title('Distribution of Air Quality by Median Household Income Bracket')
plt.xlabel('Median Household Income Bracket')
plt.ylabel('Air Quality (PM25 Mean Median)')
plt.show()

## Interpretting median household income vs. air quality
Based on the graphs, the mean median air quality is roughly similar across median household incomes between 40K and 100K with maybe a mild increase as household income rises. This could be due to economic development in areas with higher median household incomes in 1990 leading to more PM2 in the air in the 2000s. Air quality appears to have the widest variance for counties with 40-60K median incomes. Between 60-100K we see more outliers though. Further investigation may be needed to understand if a relationship exists between the two.

We will now consider air quality and outcomes. Are there there are counties where participants ended up with higher household incomes than the median household income of the county they grew up in when they were kids and if so, is there a more clear relationship between air quality? Let's compare these variables.

In [ ]:
county_data['hhic_better_than_median'] = county_data['Household_Income_at_Age_35'] > county_data['Median_Household_Income_1990']
county_data['hhic_better_than_median'].value_counts()

In [ ]:
data.head(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Create a plot for each year (2001 to 2006)
years = [2001, 2003, 2005, 2007, 2009, 2011]
for i, year in enumerate(years):
    ax = axes[i // 3, i % 3]
    year_data = pm_concentrations[pm_concentrations['year'] == year]
    data = pd.merge(county_data, year_data, on='county')
    sns.scatterplot(data=data, x='pm25_mean_median',
                    y='Individual_Income_Excluding_Spouse_at_Age_35', hue='hhic_better_than_median', ax=ax)
    ax.set_title(f'Air Quality vs Individual Income ({year})')
    ax.set_xlabel('Air Quality (PM25 Mean Median)')
    ax.set_ylabel('Individual Income at Age 35')

# Adjust layout to avoid overlap
plt.tight_layout()

# Show the plots
plt.show()

## Interpretting our air quality vs mobility across the years plots
Without factoring in median household incomes in 1990, the graphs indicate a vertical and horizontal relationship between air quality and individual income outcomes during the earliest years of collection, with scatterplots becoming more sparse, noisier over time as PM2 decreases overall. It appears that in 2001, lower observations of air quality pollutions were associated with a wider range of income outcomes. What accounts for this difference? As air quality increases, the scatter is almost flat, though positively rising slightly. This brings to question can county air quality still act as a strong predictor of outcomes?

Comparing outcomes where household income was better than median household income of the county participants grew up in, we see that they consistantly score in the lower air quality range. Is this coincidental? Further investigation is warranted in which we may need to consider what outcomes air quality may be able to predict and how other characteristics may influence this perceived effect.

# Conclusion:
Categorical: **years, hhic_better_than_median**, Numerical: **air pollutants (PM2)**, **individual household income**.
EDA indicates that we have 56 counties in our dataset once NA rows are removed. Additional data may help us better reach a conclusion around the relationship, causal or not, between air quality and outcomes and whether air quality can predict them.
Further, we found a strong positive correlation between median household income in 1990 and individual household income outcomes and median household income in 1990 and college graduation rate across counties and noisier relationships between other neighborhood characteristics and these two outcomes. We also found the data between air quality and outcomes and neighborhood characters to have an unclear relationship. However, when we only considered improvements in household income, we saw that counties with these outcomes were at the lowest range of air quality. This doesn't rule out a causal relationship, but indicates that additional investigation will be needed to draw a conclusion.